In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv


class RoomGraphEncoder(nn.Module):

    def __init__(self, input_dim=8, hidden_dim=64, output_dim=128):
        super().__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        return x

In [4]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# Node features:
# [area, mass, cabin, corridor, kitchen, lounge, engine, storage]
x = torch.tensor([
    [0.50, 0.17, 1, 0, 0, 0,  0,0],  # 0: Cabin 1
    [0.67, 0.25, 0, 1, 0, 0, 0, 0],  # 1: Corridor
    [0.47, 0.16, 1, 0, 0, 0, 0, 0],  # 2: Cabin 2
    [0.60, 0.38, 0, 0, 1, 0, 0, 0],  # 3: Kitchen
    [1.00, 0.42, 0, 0, 0, 1, 0, 0],  # 4: Lounge
    [0.83, 1.00, 0, 0, 0, 0, 1, 0],  # 5: Engine room
    [0.40, 0.21, 0, 0, 0, 0, 0, 1],  # 6: Storage
], dtype=torch.float)


# Define edges (bidirectional)
edge_index = torch.tensor([
    [0, 1, 1, 1, 1, 2, 3, 4, 4, 5, 5, 6],
    [1, 0, 2, 3, 4, 1, 1, 1, 5, 4, 6, 5]
], dtype=torch.long)

# Create PyG graph
graph = Data(
    x=x,
    edge_index=edge_index
)

print(graph)
print("Node features:", graph.x.shape)
print("Edges:", graph.edge_index.shape)

model = RoomGraphEncoder(input_dim=8, hidden_dim=64, output_dim=128)
model(graph.x, graph.edge_index).shape

Data(x=[7, 8], edge_index=[2, 12])
Node features: torch.Size([7, 8])
Edges: torch.Size([2, 12])


torch.Size([7, 128])